# Mínimos quadrados e ajuste de modelos

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/07-minimos-quadrados.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. formular um ajuste de modelo como um problema de mínimos quadrados $\min\|A\mathbf{x} - \mathbf{b}\|$;
> 2. interpretar a solução como uma **projeção** e deduzir as equações normais;
> 3. escolher entre equações normais, QR e SVD, sabendo o que cada uma faz com o condicionamento;
> 4. avaliar um ajuste pelos **resíduos**, e não só pelo $R^2$, e escolher a complexidade do modelo com validação;
> 5. ajustar modelos não lineares nos parâmetros (Gauss-Newton) sem linearizações que distorcem o ruído.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy.optimize import brentq, curve_fit, least_squares
from cnlivro.ajuste import equacoes_normais, minimos_quadrados_qr, r2, gauss_newton

## 1. Problema de engenharia: calibrar um termopar

Um termopar tipo K gera uma tensão $E$ de alguns milivolts que depende da temperatura da junta. Para usá-lo num forno que trabalha entre 0 e $500\ ^\circ$C, precisamos da **curva de calibração** $T(E)$, com erro máximo de $\pm 0{,}5\ ^\circ$C.

No laboratório, o termopar foi colocado num banho de temperatura controlada em 21 temperaturas, de 25 em $25\ ^\circ$C, e a tensão foi medida com um voltímetro de resolução de microvolts. Toda medida tem ruído: cerca de $2\ \mu$V na tensão e $0{,}05\ ^\circ$C na temperatura do banho.

In [ ]:
# Função de referência do termopar tipo K (polinômio inverso da norma ITS-90, 0 a 500 °C), usada para simular o ensaio
d_ref = [0, 2.508355e+01, 7.860106e-02, -2.503131e-01, 8.315270e-02, -1.228034e-02,
         9.804036e-04, -4.413030e-05, 1.057734e-06, -1.052755e-08]
T_de_E = np.polynomial.Polynomial(d_ref)              # T em °C, E em mV

rng = np.random.default_rng(7)
T_banho = np.linspace(0, 500, 21)
E_verdadeira = np.array([brentq(lambda e: T_de_E(e) - T, -0.1, 21.0) for T in T_banho])
E_med = E_verdadeira + rng.normal(0, 0.002, T_banho.size)       # ruído de 2 µV
T_med = T_banho + rng.normal(0, 0.05, T_banho.size)              # ruído de 0,05 °C
for E, T in zip(E_med[::4], T_med[::4]):
    print(f"E = {E:7.3f} mV    T = {T:7.2f} °C")
print("...")

São 21 pontos com ruído, e queremos uma função **simples** que passe **perto** de todos eles, sem a obrigação de passar exatamente por cada um. Interpolar, como no capítulo 6, copiaria o ruído. O que queremos é **ajustar** um modelo.

## 2. A ideia em uma imagem

Com três medidas $(t_i, y_i)$ e uma reta $y = a_0 + a_1 t$, temos três equações e duas incógnitas: em geral, nenhuma reta passa pelos três pontos. Os mínimos quadrados escolhem a reta que minimiza a soma dos quadrados dos **resíduos**, as distâncias verticais da figura da esquerda.

À direita está a mesma coisa vista de outro ângulo. O vetor das medidas, $\mathbf{b} = (y_1, y_2, y_3)$, é um ponto do espaço de três dimensões. Todas as retas possíveis geram vetores $A\mathbf{x} = a_0(1, 1, 1) + a_1(t_1, t_2, t_3)$, que formam um **plano**. A melhor reta é a **projeção ortogonal** de $\mathbf{b}$ nesse plano, e o resíduo é perpendicular a ele.

In [ ]:
t3 = np.array([0.0, 1.0, 2.0])
b3 = np.array([1.0, 3.0, 2.2])
A3 = np.column_stack([np.ones(3), t3])
x3 = np.linalg.lstsq(A3, b3, rcond=None)[0]
p3 = A3 @ x3
fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58], horizontal_spacing=0.06,
                    specs=[[{"type": "xy"}, {"type": "scene"}]],
                    subplot_titles=("os resíduos", "a projeção de b no plano das colunas de A"))
tt = np.linspace(-0.3, 2.3, 2)
fig.add_trace(go.Scatter(x=tt, y=x3[0] + x3[1] * tt, mode="lines", name="reta ajustada", line=dict(color=CORES["aproximacao"])), 1, 1)
for ti, bi, pi in zip(t3, b3, p3):
    fig.add_trace(go.Scatter(x=[ti, ti], y=[bi, pi], mode="lines", showlegend=False, line=dict(color=CORES["erro"], width=3)), 1, 1)
fig.add_trace(go.Scatter(x=t3, y=b3, mode="markers", name="medidas", marker=dict(color=CORES["exata"], size=10)), 1, 1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name="resíduos", line=dict(color=CORES["erro"], width=3)), 1, 1)
fig.update_xaxes(title="t", row=1, col=1)
fig.update_yaxes(title="y", row=1, col=1)
# plano gerado por (1,1,1) e (0,1,2)
s, u = np.meshgrid(np.linspace(-0.5, 3.5, 2), np.linspace(-0.6, 1.2, 2))
P = s[..., None] * A3[:, 0] + u[..., None] * A3[:, 1]
fig.add_trace(go.Surface(x=P[..., 0], y=P[..., 1], z=P[..., 2], opacity=0.35, showscale=False,
                         colorscale=[[0, CORES["aproximacao"]], [1, CORES["aproximacao"]]], hoverinfo="skip",
                         name="col(A)"), 1, 2)
def seta(p, q, cor, nome, largura=6):
    fig.add_trace(go.Scatter3d(x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]], mode="lines+markers",
                               marker=dict(size=[0, 5], color=cor), line=dict(color=cor, width=largura), name=nome), 1, 2)
seta(np.zeros(3), b3, CORES["exata"], "b (medidas)")
seta(np.zeros(3), p3, CORES["aproximacao"], "A x̂ (projeção)")
seta(p3, b3, CORES["erro"], "resíduo r = b − A x̂")
fig.update_scenes(xaxis=dict(title="y₁", showticklabels=False), yaxis=dict(title="y₂", showticklabels=False),
                  zaxis=dict(title="y₃", showticklabels=False), aspectmode="cube",
                  camera=dict(eye=dict(x=1.9, y=0.5, z=0.5)))
fig.update_layout(width=980, height=470, legend=dict(orientation="h", y=-0.16), margin=dict(t=60))
mostrar(fig)

**O que notar:** o resíduo $\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$ é perpendicular ao plano. Essa única condição geométrica dá todas as fórmulas da seção 3. Na figura 3D, que dá para girar com o mouse, o segmento vermelho é a distância de $\mathbf{b}$ ao plano, e a soma dos quadrados dos resíduos é o quadrado do comprimento dele.

## 3. Formulação matemática

### 3.1 Modelos lineares nos parâmetros

Um modelo é **linear nos parâmetros** quando é uma combinação linear de funções conhecidas:

$$
y(t) = x_1 \varphi_1(t) + x_2 \varphi_2(t) + \cdots + x_n \varphi_n(t).
$$

As funções $\varphi_j$ podem ser tão não lineares quanto se queira em $t$. Exemplos: o polinômio ($\varphi_j = t^{j-1}$), a equação de Steinhart-Hart dos termistores ($1/T = x_1 + x_2 \ln R + x_3 (\ln R)^3$) e uma soma de senos e cossenos. Com $m$ medidas $(t_i, y_i)$, as equações $y(t_i) = y_i$ formam um sistema **sobredeterminado**, com $m > n$:

$$
A\mathbf{x} \approx \mathbf{b}, \qquad A_{ij} = \varphi_j(t_i), \quad b_i = y_i .
$$

### 3.2 Equações normais

Os mínimos quadrados escolhem $\mathbf{x}$ que minimiza a soma dos quadrados dos resíduos,

$$
S(\mathbf{x}) = \|\mathbf{b} - A\mathbf{x}\|^2 = \sum_{i=1}^{m} \big(y_i - y(t_i)\big)^2 .
$$

O gradiente de $S$ é $-2A^{\mathsf T}(\mathbf{b} - A\mathbf{x})$. Igualá-lo a zero dá as **equações normais**:

$$
A^{\mathsf T}A\,\hat{\mathbf{x}} = A^{\mathsf T}\mathbf{b}.
$$ (eq-normais)

A interpretação geométrica é a da figura: $A^{\mathsf T}\mathbf{r} = \mathbf{0}$ diz que o resíduo é ortogonal a todas as colunas de $A$. A matriz $A^{\mathsf T}A$ é $n \times n$, simétrica e positiva definida se as colunas de $A$ forem independentes, e então a solução é única.

Para a reta $y = a_0 + a_1 t$, as equações normais dão as fórmulas da regressão linear:

$$
a_1 = \frac{m\sum t_i y_i - \sum t_i \sum y_i}{m\sum t_i^2 - \left(\sum t_i\right)^2},
\qquad
a_0 = \bar y - a_1 \bar t .
$$ (eq-regressao)

### 3.3 Como resolver: o condicionamento importa

As equações normais são simples, mas têm um defeito grave:

$$
\kappa(A^{\mathsf T}A) = \kappa(A)^2 .
$$

Pela regra do capítulo 4, resolvê-las perde cerca de $2\log_{10}\kappa(A)$ algarismos, o **dobro** do que o problema exige. Há duas alternativas que trabalham com $A$ diretamente:

- **QR.** Escreve $A = QR$, com $Q$ de colunas ortonormais e $R$ triangular superior. Como $Q$ preserva comprimentos, $\|\mathbf{b} - A\mathbf{x}\| $ é mínimo quando $R\mathbf{x} = Q^{\mathsf T}\mathbf{b}$, que se resolve por substituição regressiva. O erro é proporcional a $\kappa(A)$, não a $\kappa(A)^2$. É o método padrão.
- **SVD.** Escreve $A = U\Sigma V^{\mathsf T}$ (capítulo 8) e dá $\hat{\mathbf{x}} = V\Sigma^{+}U^{\mathsf T}\mathbf{b}$. É o mais caro e o mais robusto: funciona mesmo quando as colunas de $A$ são quase dependentes, porque permite descartar os valores singulares minúsculos. É o que `np.linalg.lstsq` usa.

Igualmente importante é a **escolha da base**. A base $1, t, t^2, \ldots$ com $t$ em milivolts, ou pior em microvolts, produz colunas com escalas muito diferentes e quase paralelas. Basta mudar a variável para $s \in [-1, 1]$, como faz `np.polynomial.Polynomial.fit`, para reduzir $\kappa(A)$ em muitas ordens de grandeza.

### 3.4 Qualidade do ajuste

O **coeficiente de determinação**

$$
R^2 = 1 - \frac{\sum r_i^2}{\sum (y_i - \bar y)^2}
$$

mede a fração da variação dos dados que o modelo explica. Ele é popular e **perigoso**: quando os dados variam muito (de 0 a $500\ ^\circ$C), qualquer modelo razoável tem $R^2$ perto de 1, mesmo errando vários graus. O que diz se o modelo serve é:

- o **erro padrão** $s = \sqrt{\sum r_i^2/(m - n)}$, na unidade da medida, a comparar com a precisão exigida e com o ruído esperado;
- o **gráfico dos resíduos**: se o modelo é adequado, os resíduos parecem ruído, sem padrão. Uma curva sistemática nos resíduos indica que falta algo no modelo;
- a **incerteza dos parâmetros**, dada pela matriz de covariância $s^2 (A^{\mathsf T}A)^{-1}$.

### 3.5 Quantos parâmetros?

Mais parâmetros sempre diminuem $\sum r_i^2$: com $n = m$ o modelo interpola e o resíduo é zero. Mas, a partir de certo ponto, o modelo passa a ajustar o **ruído**, e as previsões entre os pontos pioram. É o **sobreajuste** (*overfitting*). A forma honesta de escolher $n$ é a **validação**: separar alguns pontos, ajustar com os outros e medir o erro nos separados (seção 5).

### 3.6 Modelos não lineares nos parâmetros

No resfriamento de um corpo, $T(t) = T_\infty + A\,e^{-t/\tau}$, o parâmetro $\tau$ aparece dentro da exponencial. Ainda queremos minimizar $\|\mathbf{r}(\mathbf{p})\|^2$, com $r_i(\mathbf{p}) = T(t_i; \mathbf{p}) - T_i$, mas agora $\mathbf{r}$ não é linear em $\mathbf{p}$. O método de **Gauss-Newton** lineariza $\mathbf{r}$ em torno do chute atual e resolve um problema linear de mínimos quadrados por passo:

$$
J(\mathbf{p}_k)\,\Delta\mathbf{p} \approx -\mathbf{r}(\mathbf{p}_k), \qquad J_{ij} = \frac{\partial r_i}{\partial p_j}.
$$

É o método de Newton do capítulo 5 sem as derivadas segundas. O **Levenberg-Marquardt** acrescenta um amortecimento que o torna robusto longe da solução. É o que `scipy.optimize.least_squares` e `curve_fit` usam.

A tentação de "linearizar" o modelo, tomando o logaritmo, troca o problema por outro e distorce o ruído (seção 7.3).

## 4. À mão

Uma mola foi esticada de $x$ cm e a força medida em newtons:

| $x_i$ (cm) | 0 | 1 | 2 | 3 | 4 |
| --- | --- | --- | --- | --- | --- |
| $F_i$ (N) | 0,2 | 2,1 | 3,8 | 6,1 | 7,9 |

Com $m = 5$: $\sum x_i = 10$, $\sum F_i = 20{,}1$, $\sum x_i^2 = 30$ e $\sum x_i F_i = 59{,}6$. Por {eq}`eq-regressao`:

$$
a_1 = \frac{5 \cdot 59{,}6 - 10 \cdot 20{,}1}{5 \cdot 30 - 10^2} = \frac{298 - 201}{50} = 1{,}94\ \text{N/cm},
\qquad
a_0 = \frac{20{,}1}{5} - 1{,}94 \cdot \frac{10}{5} = 0{,}14\ \text{N}.
$$

Os resíduos $F_i - (a_0 + a_1 x_i)$ são $0{,}06$; $0{,}02$; $-0{,}22$; $0{,}14$ e $0{,}00$. A soma deles é zero, como deve ser num modelo com termo constante. A soma dos quadrados é $0{,}072$, o erro padrão é $s = \sqrt{0{,}072/3} = 0{,}15$ N e $R^2 = 0{,}998$. A rigidez da mola é $k \approx 194$ N/m.

In [ ]:
x = np.array([0, 1, 2, 3, 4.0])
F = np.array([0.2, 2.1, 3.8, 6.1, 7.9])
A = np.column_stack([np.ones_like(x), x])
coef = minimos_quadrados_qr(A, F)
residuos = F - A @ coef
print("a0, a1 =", coef.round(4))
print("resíduos:", residuos.round(3), "  soma:", round(residuos.sum(), 12))
print("A^T r =", (A.T @ residuos).round(12), " (o resíduo é ortogonal às colunas)")
print("R² =", round(r2(F, A @ coef), 4))

## 5. Explore

Os pontos azuis são 15 medidas com ruído de uma função suave, desenhada em preto. Os pontos laranja foram **separados** para validação e não entram no ajuste. Arraste o slider para mudar o grau do polinômio ajustado e observe os dois erros no título.

In [ ]:
rng5 = np.random.default_rng(2)
x_tr = np.sort(np.r_[0.0, 1.0, rng5.uniform(0, 1, 13)])     # o ajuste cobre o intervalo todo
x_va = np.sort(rng5.uniform(0.03, 0.97, 10))               # a validação fica dentro dele
f_suave = lambda x: np.sin(2 * np.pi * x) + 0.5 * x
y_tr = f_suave(x_tr) + rng5.normal(0, 0.15, x_tr.size)
y_va = f_suave(x_va) + rng5.normal(0, 0.15, x_va.size)
xx = np.linspace(0, 1, 400)

def rms(r):
    return np.sqrt(np.mean(r**2))

def figura_grau(g):
    p = np.polynomial.Polynomial.fit(x_tr, y_tr, g)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xx, y=f_suave(xx), name="função verdadeira", line=dict(color=CORES["exata"], dash="dot")))
    fig.add_trace(go.Scatter(x=xx, y=p(xx), name=f"ajuste de grau {g}", line=dict(color=CORES["aproximacao"])))
    fig.add_trace(go.Scatter(x=x_tr, y=y_tr, mode="markers", name="ajuste (15 pontos)", marker=dict(color=CORES["aproximacao"], size=8)))
    fig.add_trace(go.Scatter(x=x_va, y=y_va, mode="markers", name="validação (10 pontos)",
                             marker=dict(color=CORES["terciaria"], size=9, symbol="diamond")))
    fig.update_xaxes(title="x")
    fig.update_yaxes(title="y", range=[-2, 2.5])
    fig.update_layout(title=f"grau {g}: erro RMS no ajuste {numero(rms(p(x_tr) - y_tr), 3)}, na validação {numero(rms(p(x_va) - y_va), 3)}",
                      height=420)
    return fig

graus = list(range(0, 15))
quadros = [(str(g), figura_grau(g)) for g in graus]
mostrar(com_controle(quadros, rotulo="grau = ", ativo=10, titulos=[q[1].layout.title.text for q in quadros]))

O gráfico a seguir resume o slider: o erro no ajuste só cai, mas o erro na validação tem um mínimo.

In [ ]:
erro_tr = [rms(np.polynomial.Polynomial.fit(x_tr, y_tr, g)(x_tr) - y_tr) for g in graus]
erro_va = [rms(np.polynomial.Polynomial.fit(x_tr, y_tr, g)(x_va) - y_va) for g in graus]
fig = go.Figure()
fig.add_trace(go.Scatter(x=graus, y=erro_tr, mode="lines+markers", name="nos pontos do ajuste", line=dict(color=CORES["aproximacao"])))
fig.add_trace(go.Scatter(x=graus, y=erro_va, mode="lines+markers", name="nos pontos de validação", line=dict(color=CORES["terciaria"])))
fig.add_hline(y=0.15, line=dict(color=CORES["referencia"], dash="dash"), annotation_text="nível do ruído", annotation_position="top right")
fig.update_xaxes(title="grau do polinômio")
fig.update_yaxes(type="log", title="erro RMS")
fig.update_layout(title="Mais parâmetros: o ajuste melhora, a previsão piora")
mostrar(fig)
print("grau com menor erro de validação:", graus[int(np.argmin(erro_va))])

**O que notar:** até o grau 3, os dois erros caem juntos, porque o modelo está aprendendo a forma da função. Entre os graus 3 e 7, os dois ficam perto do nível do ruído: qualquer um deles é um bom modelo, e o mais simples é o preferível. A partir do grau 8, o erro no ajuste continua caindo **abaixo do nível do ruído**, sinal de que o modelo está decorando o ruído, e o erro de validação dispara. No grau 14, o polinômio interpola os 15 pontos e oscila entre eles, como no capítulo 6.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Tenho dados de calibração de um termopar: tensões `E_med` (mV) e temperaturas `T_med` (°C). Ajuste uma curva de calibração T(E) por mínimos quadrados e me diga se ela atende ao requisito de erro de ±0,5 °C. Use o R² para avaliar a qualidade do ajuste."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
coef_cal = np.polyfit(E_med, T_med, 1)                # T = a·E + b
T_ajustada = np.polyval(coef_cal, E_med)
R2 = 1 - np.sum((T_med - T_ajustada) ** 2) / np.sum((T_med - T_med.mean()) ** 2)
print(f"T = {coef_cal[0]:.4f}·E + {coef_cal[1]:.4f}")
print(f"R² = {R2:.6f}")
print("Calibração aprovada: o ajuste explica mais de 99,99 % da variação." if R2 > 0.9999 else "Calibração reprovada.")

O agente escolheu uma reta, calculou o $R^2$ como pedido e aprovou a calibração. O pedido, porém, falava em **erro de $\pm 0{,}5\ ^\circ$C**, e o $R^2$ não mede isso.

### 6.2 Verificação: os resíduos

In [ ]:
residuos_reta = T_med - T_ajustada
print(f"maior resíduo: {np.abs(residuos_reta).max():.2f} °C;  erro padrão s = {np.sqrt(np.sum(residuos_reta**2) / (len(T_med) - 2)):.2f} °C")

fig = make_subplots(rows=1, cols=2, subplot_titles=("ajuste: medidas e reta", "resíduos da reta"), horizontal_spacing=0.1)
fig.add_trace(go.Scatter(x=E_med, y=T_med, mode="markers", name="medidas", marker=dict(color=CORES["exata"], size=7)), 1, 1)
fig.add_trace(go.Scatter(x=E_med, y=T_ajustada, mode="lines", name="reta", line=dict(color=CORES["aproximacao"])), 1, 1)
fig.add_trace(go.Scatter(x=E_med, y=residuos_reta, mode="markers+lines", name="resíduos", line=dict(color=CORES["erro"])), 1, 2)
for lim in (-0.5, 0.5):
    fig.add_hline(y=lim, line=dict(color=CORES["destaque"], dash="dash", width=1), row=1, col=2)
fig.update_xaxes(title="E (mV)")
fig.update_yaxes(title="T (°C)", row=1, col=1)
fig.update_yaxes(title="resíduo (°C)", row=1, col=2)
fig.update_layout(width=900, height=400, showlegend=False)
mostrar(fig)

**O que notar:** à esquerda, a reta parece perfeita, e é isso que o $R^2$ "vê". À direita, os resíduos formam uma curva **sistemática**, com erros de quase $3\ ^\circ$C, seis vezes o permitido. A faixa verde é o requisito. Os resíduos não parecem ruído: a relação entre tensão e temperatura do termopar não é linear, e o modelo está errado.

O $R^2$ é alto porque a temperatura varia 500 graus e a reta explica quase toda essa variação. Mas 3 graus em 500 é exatamente o tipo de erro que o projeto não tolera.

### 6.3 A correção

Aumentamos o grau, usando a base escalada de `np.polynomial.Polynomial.fit` (seção 3.3), e escolhemos pelo critério que importa: o maior resíduo, comparado com o requisito e com o ruído. Como o ruído de $2\ \mu$V vale cerca de $0{,}05\ ^\circ$C, um bom modelo deve deixar resíduos dessa ordem e **sem padrão**.

In [ ]:
print("grau   R²           maior resíduo (°C)   s (°C)")
ajustes = {}
for grau in range(1, 9):
    p = np.polynomial.Polynomial.fit(E_med, T_med, grau)
    r = T_med - p(E_med)
    ajustes[grau] = p
    print(f"{grau:3d}   {r2(T_med, p(E_med)):.8f}   {np.abs(r).max():18.3f}   {np.sqrt(np.sum(r**2) / (len(T_med) - grau - 1)):6.3f}")

O $R^2$ é praticamente o mesmo em todas as linhas, e por isso não ajuda a escolher. O maior resíduo cai abaixo de $0{,}5\ ^\circ$C a partir do grau 5, e no grau 6 o erro padrão já está perto do nível do ruído. Graus maiores quase não melhoram e começam a correr o risco de sobreajuste. Conferimos os resíduos do grau 6 e o erro **entre** os pontos, contra a função de referência, que aqui conhecemos porque os dados foram simulados:

In [ ]:
p6 = ajustes[6]
E_fino = np.linspace(E_med.min(), E_med.max(), 500)
erro_entre = np.abs(p6(E_fino) - T_de_E(E_fino)).max()
fig = go.Figure()
fig.add_trace(go.Scatter(x=E_med, y=T_med - p6(E_med), mode="markers+lines", name="resíduos do grau 6", line=dict(color=CORES["aproximacao"])))
fig.add_trace(go.Scatter(x=E_med, y=residuos_reta, mode="markers+lines", name="resíduos da reta", line=dict(color=CORES["erro"]), opacity=0.4))
for lim in (-0.5, 0.5):
    fig.add_hline(y=lim, line=dict(color=CORES["destaque"], dash="dash", width=1))
fig.update_xaxes(title="E (mV)")
fig.update_yaxes(title="resíduo (°C)", range=[-1.2, 1.2])
fig.update_layout(title="Grau 6: resíduos dentro do requisito e sem padrão claro")
mostrar(fig)
print(f"erro máximo entre os pontos, contra a referência: {erro_entre:.2f} °C")

> **Lição.** $R^2$ alto não valida um modelo. Avalie um ajuste pelos resíduos, na unidade do problema e contra o requisito, e procure padrões neles. Quando pedir um ajuste a um agente, diga o critério de aceitação que importa, e não um indicador genérico.

## 7. Quando falha

### 7.1 As equações normais perdem o dobro dos algarismos

A matriz de Läuchli é o exemplo clássico. Com $\delta = 10^{-8}$,

$$
A = \begin{bmatrix} 1 & 1 \\ \delta & 0 \\ 0 & \delta \end{bmatrix},
\qquad
A^{\mathsf T}A = \begin{bmatrix} 1 + \delta^2 & 1 \\ 1 & 1 + \delta^2 \end{bmatrix}.
$$

Como $\delta^2 = 10^{-16}$ é menor que $\varepsilon_{\text{mach}}$, o computador guarda $1 + \delta^2$ como 1, e $A^{\mathsf T}A$ fica **exatamente singular**, embora $\kappa(A) \approx 1{,}4 \times 10^8$ seja razoável:

In [ ]:
dlt = 1e-8
A = np.array([[1, 1], [dlt, 0], [0, dlt]])
b = A @ np.array([1.0, 1.0])
print(f"κ(A) = {np.linalg.cond(A):.1e}")
print("QR:", minimos_quadrados_qr(A, b))
try:
    print("equações normais:", equacoes_normais(A, b))
except np.linalg.LinAlgError as e:
    print("equações normais: falharam (", e, ")")

### 7.2 Um ponto ruim

Os mínimos quadrados elevam os resíduos ao quadrado, então um único ponto muito errado (um *outlier*, por exemplo um erro de digitação) puxa o ajuste inteiro. Uma **função de perda robusta**, que cresce mais devagar para resíduos grandes, reduz esse efeito:

In [ ]:
F_ruim = F.copy()
F_ruim[3] = 16.1                                            # 6,1 digitado como 16,1
A = np.column_stack([np.ones_like(x), x])
quadrados = minimos_quadrados_qr(A, F_ruim)
robusto = least_squares(lambda c: A @ c - F_ruim, x0=[0, 1], loss="soft_l1", f_scale=0.3).x
print(f"sem o erro:            k = {minimos_quadrados_qr(A, F)[1] * 100:.0f} N/m")
print(f"mínimos quadrados:     k = {quadrados[1] * 100:.0f} N/m")
print(f"perda robusta soft_l1: k = {robusto[1] * 100:.0f} N/m")

O melhor remédio continua sendo **olhar os resíduos**: um outlier aparece como um ponto isolado muito longe dos outros.

### 7.3 Linearizar distorce o ruído

Para ajustar o resfriamento $T(t) = T_\infty + A\,e^{-t/\tau}$ com $T_\infty = 25\ ^\circ$C conhecida, é tentador tomar o logaritmo, $\ln(T - T_\infty) = \ln A - t/\tau$, e ajustar uma reta. O problema é que o ruído das medidas é aditivo em $T$. Quando $T - T_\infty$ fica pequeno, no fim do resfriamento, o logaritmo amplifica o ruído, e os pontos com $T - T_\infty \le 0$ nem podem ser usados:

In [ ]:
rng7 = np.random.default_rng(5)
t_r = np.linspace(0, 120, 41)
T_amb, tau_real = 25.0, 15.0
T_r = T_amb + 65.0 * np.exp(-t_r / tau_real) + rng7.normal(0, 0.5, t_r.size)

positivos = T_r - T_amb > 0
reta = np.polyfit(t_r[positivos], np.log(T_r[positivos] - T_amb), 1)
tau_lin = -1 / reta[0]

residuo = lambda p: T_amb + p[0] * np.exp(-t_r / p[1]) - T_r
jacobiana = lambda p: np.column_stack([np.exp(-t_r / p[1]), p[0] * t_r / p[1]**2 * np.exp(-t_r / p[1])])
gn = gauss_newton(residuo, jacobiana, [np.exp(reta[1]), tau_lin])
print(f"τ real: {tau_real} s")
print(f"linearizando: τ = {tau_lin:.2f} s   ({(~positivos).sum()} pontos descartados por T − T∞ ≤ 0)")
print(f"Gauss-Newton: τ = {gn.x[1]:.2f} s   ({gn.iteracoes} iterações, partindo do resultado linearizado)")

A linearização erra $\tau$ em cerca de 15 %. Ela serve como **chute inicial**, que é como foi usada aqui, mas a resposta deve vir do ajuste não linear, que mede os resíduos na unidade em que o ruído realmente está.

### 7.4 Extrapolar uma calibração

A curva de grau 6 foi ajustada entre 0 e $500\ ^\circ$C. Fora desse intervalo, os termos de grau alto dominam:

In [ ]:
for E in [22.0, 24.0, 26.0]:
    print(f"E = {E:4.1f} mV:  grau 6 dá {p6(E):7.1f} °C   (a reta daria {np.polyval(coef_cal, E):6.1f} °C)")
print("Pela tabela do termopar tipo K, 22, 24 e 26 mV correspondem a cerca de 532, 579 e 626 °C.")

A 26 mV, a curva de grau 6 erra mais de $35\ ^\circ$C, e até a reta, reprovada dentro do intervalo, erra menos. Uma calibração vale só no intervalo calibrado. Para estender o intervalo, calibre de novo.

## Resumo

| Ideia | Em uma frase |
| --- | --- |
| Mínimos quadrados | minimizar $\|A\mathbf{x} - \mathbf{b}\|^2$; o resíduo fica ortogonal às colunas de $A$ |
| Equações normais | $A^{\mathsf T}A\mathbf{x} = A^{\mathsf T}\mathbf{b}$; simples, mas $\kappa$ ao quadrado |
| QR / SVD | trabalham com $A$; são os métodos usados por `lstsq` e `Polynomial.fit` |
| Base e escala | variável escalada para $[-1, 1]$; o condicionamento muda muito |
| $R^2$ | perto de 1 não prova nada; olhe os resíduos na unidade do problema |
| Sobreajuste | escolha o número de parâmetros por validação |
| Não linear | Gauss-Newton / Levenberg-Marquardt; linearizar só para o chute |

## Exercícios

**Conceituais (sem computador)**

1. Deduza {eq}`eq-regressao` a partir de {eq}`eq-normais` para a reta $y = a_0 + a_1 t$.
2. Mostre que, num modelo com termo constante, a soma dos resíduos de mínimos quadrados é zero. Por que a reta passa pelo ponto $(\bar t, \bar y)$?
3. Mostre que $\kappa_2(A^{\mathsf T}A) = \kappa_2(A)^2$ usando a SVD de $A$.
4. Escreva o modelo de Steinhart-Hart de um termistor como um problema linear de mínimos quadrados. Quais são as colunas de $A$?

**Verificação (com agente)**

5. Peça a um agente o ajuste do modelo de Vogel, $\mu = A\,e^{B/(T - C)}$, aos dados de viscosidade do capítulo 6, com `curve_fit`. Verifique os resíduos e a incerteza dos parâmetros. Compare a extrapolação a $120\ ^\circ$C com a do PCHIP.
6. Peça a um agente uma regressão linear múltipla do consumo de energia de um prédio em função da temperatura externa e da ocupação. Proponha um teste com dados fabricados, gerados por um modelo conhecido com ruído, para verificar o código.

**Projeto aberto**

7. **Calibração com incerteza.** Repita o ajuste do termopar 200 vezes, cada vez com um novo sorteio do ruído, e desenhe a faixa de variação da curva de calibração. Compare essa faixa com a incerteza prevista pela matriz de covariância $s^2(A^{\mathsf T}A)^{-1}$. O apêndice A trata de incerteza em mais detalhe.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 17 e 20.
- L. N. Trefethen e D. Bau, *Numerical Linear Algebra*, SIAM, 1997, aulas 7, 8, 11 e 18 a 19 (QR e mínimos quadrados).
- NIST, *ITS-90 Thermocouple Database*, para as funções de referência dos termopares.